# Generate Modeling Dataset

**Purpose**: Create machine learning dataset by orchestrating existing Python scripts.


## Prerequisites

1. ✅ Run `HSA_v6_FINAL.ipynb` to generate HSA boundaries
2. ✅ Run `Population_Allocation_Probabilistic_v2.ipynb`
3. ✅ Run `GEE_HSA_Weekly_Climate_Lagged.ipynb` (or a local version)
4. ✅ Make sure climate CSVs are downloaded to `{OUT_DIR}/DRIVE_CLIMATE_BY_HSA_DOWNLOAD/FINAL_HSA_CLIMATE/`

In [ ]:
# ── BOUNDARY VERSION SELECTOR ──────────────────────────────────────────────
# Choose which HSA boundary bundle to use for this analysis.
# Must match a bundle produced by HSA_FINAL.ipynb.
#   'v6'  — original greedy algorithm (no post-selection corrections)
#   'v7'  — + anchor upgrade/demotion + major-orphan promotion
#   'v8'  — + satellite bubble boundaries
BOUNDARY_VERSION = "v7"   # change as needed
# ────────────────────────────────────────────────────────────────────────────


In [ ]:
# ============================================================================
# CONFIGURATION - All parameters for the modeling dataset pipeline
# ============================================================================
from pathlib import Path
import os
import subprocess
import sys

# --- Core Settings ---
NETWORK = "INF"                        # Network type: INF, NCD, SYNINF, SYNNCD
HSA_MODE = "footprint"                 # HSA optimization mode: fewest, footprint, distance, etc.
DISEASE_FOCUS = "diarrheal" if NETWORK in ("INF", "SYNINF") else "hypertension"

# --- Date Ranges ---
# Weekly disease counts date range (patient data aggregation window)
WEEK_START = "2019-01-07"              # Start date for weekly aggregation
WEEK_END = "2024-01-29"                # End date for weekly aggregation

# ML dataset date range (post diagnostic code change)
ML_START_DATE = "2022-06-27"           # Start of valid data for modeling
ML_END_DATE = "2024-01-29"             # End of valid data for modeling

# --- Data Quality Thresholds ---
MISSING_THRESHOLD = 0.40               # Drop features with >40% missing data
CORRELATION_THRESHOLD = 0.95           # Remove features with r > 0.95

# --- Directory Paths ---
DATA_DIR = Path(os.environ.get("HSA_DATA_DIR", "data"))
OUT_DIR = Path(os.environ.get("HSA_OUT_DIR", os.environ.get("PIPELINE_OUT_DIR", "out")))
CLIMATE_DIR = OUT_DIR / f"DRIVE_CLIMATE_BY_HSA_DOWNLOAD_{BOUNDARY_VERSION.upper()}" / "FINAL_HSA_CLIMATE"
MODELING_DIR = OUT_DIR / "modeling"
MODELING_DIR.mkdir(exist_ok=True, parents=True)

print("="*80)
print("MODELING DATASET GENERATION - CONFIGURATION")
print("="*80)
print(f"Network: {NETWORK}")
print(f"HSA Mode: {HSA_MODE}")
print(f"Disease Focus: {DISEASE_FOCUS}")
print(f"Weekly aggregation: {WEEK_START} to {WEEK_END}")
print(f"ML dataset range: {ML_START_DATE} to {ML_END_DATE}")
print(f"Missing threshold: {MISSING_THRESHOLD*100:.0f}%")
print(f"Correlation threshold: {CORRELATION_THRESHOLD}")
print("="*80)

## Step 1: Check Prerequisites

In [ ]:
print("\nChecking prerequisites...")

required_files = {
    "HSA boundaries": OUT_DIR / f"{NETWORK}_{HSA_MODE}_hsas_{BOUNDARY_VERSION}.geojson",
    "Pixel allocations": OUT_DIR / f"pixel_allocations_{NETWORK}_{HSA_MODE}_{BOUNDARY_VERSION}.csv",
    "Patient data": Path("data") / f"{NETWORK}_patient_visits.csv",
    "Climate directory": OUT_DIR / f"DRIVE_CLIMATE_BY_HSA_DOWNLOAD_{BOUNDARY_VERSION.upper()}" / "FINAL_HSA_CLIMATE"
}

all_found = True
for name, filepath in required_files.items():
    if filepath.exists():
        if filepath.is_dir():
            num_files = len(list(filepath.glob("*.csv")))
            print(f"  ✓ {name}: {num_files} CSV files")
        else:
            size_mb = filepath.stat().st_size / (1024*1024)
            print(f"  ✓ {name}: {size_mb:.1f} MB")
    else:
        print(f"  ✗ {name}: NOT FOUND")
        all_found = False

if not all_found:
    raise FileNotFoundError("Missing required files. Complete prerequisite steps.")

print("\n✓ All prerequisites found!")

## Step 2: Generate Weekly Disease Counts

Calls existing script to aggregate patient visits by HSA and week.

In [ ]:
print("\n" + "="*80)
print("GENERATING WEEKLY DISEASE COUNTS")
print("="*80)

hsa_geojson = str(OUT_DIR / f"{NETWORK}_{HSA_MODE}_hsas_{BOUNDARY_VERSION}.geojson")

# Check for allocation details (try both naming conventions)
allocation_details = OUT_DIR / f"{NETWORK}_{HSA_MODE}_allocation_details_{BOUNDARY_VERSION}.csv"
allocation_details_alt = OUT_DIR / f"pixel_allocations_{NETWORK}_{HSA_MODE}_{BOUNDARY_VERSION}.csv"
has_allocation = allocation_details.exists() or allocation_details_alt.exists()

# Try adjusted script first, fallback to regular
if Path("generate_weekly_disease_counts_adjusted.py").exists() and has_allocation:
    script = "generate_weekly_disease_counts_adjusted.py"
    which_file = allocation_details if allocation_details.exists() else allocation_details_alt
    print(f"Using adjusted counts (found {which_file})")
elif Path("generate_weekly_disease_counts.py").exists():
    if Path("generate_weekly_disease_counts_adjusted.py").exists():
        print(f"Adjusted allocation file missing; falling back to unadjusted counts.")
    script = "generate_weekly_disease_counts.py"
else:
    raise FileNotFoundError("Weekly disease counts script not found")

print(f"\nUsing: {script}")
print(f"  Week range: {WEEK_START} to {WEEK_END}\n")

result = subprocess.run(
    [sys.executable, script, hsa_geojson,
     "--out-dir", str(OUT_DIR),
     "--disease", DISEASE_FOCUS,
     "--week-start", WEEK_START,
     "--week-end", WEEK_END,
     "--boundary-version", BOUNDARY_VERSION],
    capture_output=False
)

if result.returncode != 0:
    raise RuntimeError(f"Script failed with return code: {result.returncode}")

print("\n✓ Weekly disease counts generated!")

## Step 3: Merge Climate Data with Disease Counts

Calls `prepare_ml_dataset.py` to merge climate and disease data.

In [ ]:
print("\n" + "="*80)
print("MERGING CLIMATE DATA WITH DISEASE COUNTS")
print("="*80)

if not Path("prepare_ml_dataset.py").exists():
    raise FileNotFoundError("prepare_ml_dataset.py not found")

print("\nUsing: prepare_ml_dataset.py")
print(f"  Climate dir: {CLIMATE_DIR}")
print(f"  Output dir: {MODELING_DIR}")
print(f"  Date range: {ML_START_DATE} to {ML_END_DATE}")
print(f"  Missing threshold: {MISSING_THRESHOLD}")
print(f"  Correlation threshold: {CORRELATION_THRESHOLD}\n")

result = subprocess.run(
    [sys.executable, "prepare_ml_dataset.py",
     "--network", NETWORK,
     "--hsa-mode", HSA_MODE,
     "--disease-focus", DISEASE_FOCUS,
     "--out-dir", str(OUT_DIR),
     "--climate-dir", str(CLIMATE_DIR),
     "--output-dir", str(MODELING_DIR),
     "--start-date", ML_START_DATE,
     "--end-date", ML_END_DATE,
     "--missing-threshold", str(MISSING_THRESHOLD),
     "--correlation-threshold", str(CORRELATION_THRESHOLD),
     "--boundary-version", BOUNDARY_VERSION],
    capture_output=False
)

if result.returncode != 0:
    raise RuntimeError(f"Script failed with return code: {result.returncode}")

print("\n✓ ML dataset prepared!")

## Step 4: Verify Outputs

In [ ]:
import pandas as pd
import json

print("\n" + "="*80)
print("VERIFYING OUTPUTS")
print("="*80)

# Find output files
output_patterns = [
    f"*{NETWORK}*{HSA_MODE}*weekly*.csv",
    f"*{NETWORK}*{HSA_MODE}*modeling*.csv",
    f"*{NETWORK}*{HSA_MODE}*metadata*.json"
]

found_files = []
for pattern in output_patterns:
    found_files.extend(OUT_DIR.rglob(pattern))

print("\nGenerated files:")
for filepath in sorted(set(found_files)):
    if filepath.suffix == '.csv':
        df = pd.read_csv(filepath)
        size_mb = filepath.stat().st_size / (1024*1024)
        print(f"  ✓ {filepath.name} ({len(df):,} rows, {size_mb:.2f} MB)")
    else:
        print(f"  ✓ {filepath.name}")

# Load main dataset if found
dataset_files = [f for f in found_files if 'modeling_dataset' in f.name and f.suffix == '.csv']
if dataset_files:
    print("\nDataset Summary:")
    df = pd.read_csv(dataset_files[0])
    print(f"  Rows: {len(df):,}")
    print(f"  Columns: {len(df.columns)}")
    
    if 'hsa_id' in df.columns:
        print(f"  HSAs: {df['hsa_id'].nunique()}")
    if 'week_start' in df.columns:
        print(f"  Weeks: {df['week_start'].nunique()}")
        print(f"  Date range: {df['week_start'].min()} to {df['week_start'].max()}")
    if 'diarrheal_count' in df.columns:
        print(f"\n  Diarrheal diseases:")
        print(f"    Total cases: {df['diarrheal_count'].sum():,}")
        print(f"    Mean per week: {df['diarrheal_count'].mean():.2f}")
        print(f"    Weeks with cases: {(df['diarrheal_count'] > 0).sum():,}")

print("\n" + "="*80)
print("COMPLETE")
print("="*80)
print("\nNext: Use dataset in modeling pipeline")

In [ ]:
# AUTO_NOTEBOOK_SUMMARY_V1
from pathlib import Path
from datetime import datetime
import os
import json

NOTEBOOK_NAME = "Generate_Modeling_Dataset"
NETWORK = globals().get('NETWORK', os.environ.get('NETWORK', 'INF'))
HSA_MODE = globals().get('HSA_MODE', os.environ.get('HSA_MODE', ''))

suffix = f"{NETWORK}_{HSA_MODE}" if HSA_MODE else f"{NETWORK}"

out_root = Path(globals().get('OUT_DIR', globals().get('OUT_ROOT', os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', "out")))))
summary_dir = out_root / 'textresults'
summary_dir.mkdir(parents=True, exist_ok=True)
BOUNDARY_VERSION = globals().get('BOUNDARY_VERSION', os.environ.get('BOUNDARY_VERSION', os.environ.get('PIPELINE_VERSION', 'v7')))
summary_path = summary_dir / f"{NOTEBOOK_NAME}_{suffix}_{BOUNDARY_VERSION}_results.md"

files = [p for p in out_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.png', '.pdf', '.geojson', '.parquet'}]
files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
important = files[:120]

NL = "\n"

blocks = []
blocks.append(f"# Notebook Results: {NOTEBOOK_NAME}")

meta = [
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"- Network: {NETWORK}",
]
if HSA_MODE:
    meta.append(f"- HSA mode: {HSA_MODE}")
blocks.append(NL.join(meta))

file_lines = ['## Important Output Files', '']
for p in important:
    file_lines.append(f"- `{p}`")
blocks.append(NL.join(file_lines))

nb_path = Path(f"{NOTEBOOK_NAME}.ipynb")
if nb_path.exists():
    try:
        nb_data = json.loads(nb_path.read_text())
        blocks.append('## Displayed Cell Outputs')

        for idx, cell in enumerate(nb_data.get('cells', []), start=1):
            if cell.get('cell_type') != 'code':
                continue
            outputs = cell.get('outputs', []) or []
            if not outputs:
                continue

            blocks.append(f"### Cell {idx}")

            for out in outputs:
                otype = out.get('output_type')

                if otype == 'stream':
                    text = ''.join(out.get('text', [])) if isinstance(out.get('text', []), list) else str(out.get('text', ''))
                    text = text.rstrip()
                    if text:
                        blocks.append("```text" + NL + text + NL + "```")

                elif otype in ('execute_result', 'display_data'):
                    data = out.get('data', {})
                    if 'text/markdown' in data:
                        md = ''.join(data['text/markdown']) if isinstance(data['text/markdown'], list) else str(data['text/markdown'])
                        md = md.rstrip()
                        if md:
                            blocks.append(md)
                    elif 'text/plain' in data:
                        txt = ''.join(data['text/plain']) if isinstance(data['text/plain'], list) else str(data['text/plain'])
                        txt = txt.rstrip()
                        if txt:
                            blocks.append("```text" + NL + txt + NL + "```")
                    elif 'text/html' in data:
                        html = ''.join(data['text/html']) if isinstance(data['text/html'], list) else str(data['text/html'])
                        html = html.rstrip()
                        if html:
                            blocks.append("```html" + NL + html + NL + "```")
                    elif 'image/png' in data or 'image/jpeg' in data:
                        blocks.append('_[Image output omitted in text summary]_')

                elif otype == 'error':
                    tb = out.get('traceback', []) or []
                    if tb:
                        err = NL.join(str(t) for t in tb)
                    else:
                        err = f"{out.get('ename', 'Error')}: {out.get('evalue', '')}"
                    blocks.append("```text" + NL + err + NL + "```")

    except Exception as e:
        blocks.append('## Displayed Cell Outputs')
        blocks.append(f"Could not parse notebook outputs: {e}")

summary = (NL + NL).join(b for b in blocks if b and str(b).strip()) + NL
summary_path.write_text(summary)
print(f"Saved notebook summary: {summary_path}")
